# Online News Popularity — Macro F1 (A–E)  ·  v2
**Kaggle setup:** Session options → Accelerator → **GPU T4 x2** (or P100) → add the competition data via *Add Input* → **Run All**. Output: `/kaggle/working/submission.csv`.

**What's new in v2**
1. **Adversarial validation** – checks whether train and test differ; auto-drops drifting features only if AUC > 0.60.
2. **Test-like scoring** – the test set has no NAs, so every metric is computed on complete train rows only.
3. **Ablation table** – a fast LightGBM probe scores each change on identical folds with a bootstrap paired test, so you can see what actually helps.
4. **Ordinal regressor** – A→E are ordered, so a regressor on the class index is stacked in.
5. **Multi-seed** ensemble (`N_SEEDS`) + a **component scoreboard** at the end.

No external data or pretrained models.

In [1]:
import os, glob, time, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score, log_loss, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import lightgbm as lgb, xgboost as xgb
from catboost import CatBoostClassifier
warnings.filterwarnings('ignore')

SEED = 42
N_FOLDS = 5
FAST = False                 # True = tiny smoke test only
USE_GPU = True               # False if no GPU accelerator
N_SEEDS = 7                  # seeds per model (1 seed ~ 15-30 min on a T4; lower if short on time)
MODELS = ['xgb', 'cat', 'lgb']
RUN_ABLATION = True          # ~5 min, tells you which changes help
DROP_DRIFT = True            # drop adversarial features if train/test AUC > 0.60

def find(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True) + glob.glob(f'./**/{name}', recursive=True)
    assert hits, f'{name} not found - add the data via "Add Input"'
    return hits[0]

train = pd.read_csv(find('train.csv'), na_values=['NA'])
test  = pd.read_csv(find('test.csv'),  na_values=['NA'])
print(train.shape, test.shape)
classes = sorted(train.target_popularity.unique()); K = len(classes)
y = train.target_popularity.map({c: i for i, c in enumerate(classes)}).values
print(pd.Series(y).map(dict(enumerate(classes))).value_counts(normalize=True).sort_index())

(37144, 61) (2500, 60)
A    0.046172
B    0.106208
C    0.194648
D    0.363639
E    0.289333
Name: proportion, dtype: float64


## Feature engineering

In [2]:
def add_features(df):
    d = df.copy(); eps = 1e-6
    ch = [c for c in d.columns if c.startswith('data_channel_is_')]
    d['channel_id'] = np.where(d[ch].fillna(0).sum(axis=1) == 0, -1, d[ch].fillna(0).values.argmax(axis=1))
    wd = [c for c in d.columns if c.startswith('weekday_is_')]
    d['weekday_id'] = d[wd].fillna(0).values.argmax(axis=1)
    lda = [f'LDA_0{i}' for i in range(5)]
    d['lda_argmax'] = d[lda].fillna(0).values.argmax(axis=1)
    d['lda_max'] = d[lda].max(axis=1)
    d['lda_entropy'] = -(d[lda].clip(eps, 1) * np.log(d[lda].clip(eps, 1))).sum(axis=1)
    d['log_tokens'] = np.log1p(d['n_tokens_content'])
    d['links_per_token'] = d['num_hrefs'] / (d['n_tokens_content'] + 1)
    d['self_link_ratio'] = d['num_self_hrefs'] / (d['num_hrefs'] + 1)
    d['media_total'] = d['num_imgs'] + d['num_videos']
    d['imgs_per_token'] = d['num_imgs'] / (d['n_tokens_content'] + 1)
    d['title_content_ratio'] = d['n_tokens_title'] / (d['n_tokens_content'] + 1)
    d['kw_per_title'] = d['num_keywords'] / (d['n_tokens_title'] + 1)
    d['kw_range_avg'] = d['kw_max_avg'] - d['kw_min_avg']
    d['kw_avg_x_max'] = d['kw_avg_avg'] * d['kw_max_avg']
    d['kw_avg_ratio'] = d['kw_avg_avg'] / (d['kw_max_avg'] + 1)
    d['kw_max_min_ratio'] = d['kw_max_min'] / (d['kw_avg_min'] + 1)
    for c in ['kw_avg_avg','kw_max_avg','kw_min_avg','kw_avg_max','kw_min_max','kw_max_min','kw_avg_min']:
        d[f'log_{c}'] = np.log1p(d[c].clip(lower=0))
    d['selfref_range'] = d['self_reference_max_shares'] - d['self_reference_min_shares']
    d['log_selfref_avg'] = np.log1p(d['self_reference_avg_sharess'].clip(lower=0))
    d['selfref_x_links'] = d['log_selfref_avg'] * d['num_self_hrefs']
    d['pos_neg_diff'] = d['global_rate_positive_words'] - d['global_rate_negative_words']
    d['pol_range_pos'] = d['max_positive_polarity'] - d['min_positive_polarity']
    d['pol_range_neg'] = d['max_negative_polarity'] - d['min_negative_polarity']
    d['title_sub_x_pol'] = d['title_subjectivity'] * d['title_sentiment_polarity']
    d['subj_x_sent'] = d['global_subjectivity'] * d['global_sentiment_polarity']
    d['kw_avg_x_channel'] = d['kw_avg_avg'] * (d['channel_id'] + 2)
    d['timedelta_x_kw'] = d['timedelta'] * d['kw_avg_avg'] / 1000
    return d

X_raw  = train.drop(columns=['id', 'target_popularity'])
Xt_raw = test.drop(columns=['id'])
X_fe   = add_features(X_raw)
Xt_fe  = add_features(Xt_raw)[X_fe.columns]
# IMPORTANT: test.csv has NO missing values, while ~42% of train rows have at least one (random, label-independent).
# So we train on everything, but SCORE / TUNE only on complete train rows (they look like the test set).
comp = X_raw.notna().all(axis=1).values
print(X_raw.shape, X_fe.shape, f'| complete train rows: {comp.sum()} ({comp.mean():.1%}) | NAs in test: {int(Xt_raw.isna().sum().sum())}')

(37144, 59) (37144, 92) | complete train rows: 21365 (57.5%) | NAs in test: 0


## 1) Adversarial validation (train vs test)
Done on complete train rows only: the test set has zero NAs, so including NA rows would just detect 'has NAs' (AUC≈0.73, meaningless). AUC ≈ 0.50 = same distribution. Above 0.60 → drop the top drifting features.

In [3]:
adv_X = pd.concat([X_fe[comp], Xt_fe], ignore_index=True)      # complete train rows only (test has no NAs)
adv_y = np.r_[np.zeros(comp.sum()), np.ones(len(Xt_fe))]
adv_m = lgb.LGBMClassifier(n_estimators=150 if FAST else 300, learning_rate=0.05, num_leaves=31,
                           subsample=0.8, subsample_freq=1, colsample_bytree=0.7,
                           random_state=SEED, n_jobs=-1, verbose=-1)
p = cross_val_predict(adv_m, adv_X, adv_y, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method='predict_proba')[:, 1]
adv_auc = roc_auc_score(adv_y, p)
adv_m.fit(adv_X, adv_y)
imp = pd.Series(adv_m.booster_.feature_importance('gain'), index=adv_X.columns).sort_values(ascending=False)
print(f'Adversarial AUC = {adv_auc:.3f}  (0.50 = identical distributions)')
print(imp.head(8).round(0))
drift_cols = list(imp.index[:3]) if adv_auc > 0.60 else []
print('drift features to drop:', drift_cols if DROP_DRIFT else '(DROP_DRIFT off)', '' if drift_cols else '-> no meaningful drift, nothing dropped')

Adversarial AUC = 0.531  (0.50 = identical distributions)
LDA_01                   1657.0
LDA_00                   1638.0
min_positive_polarity    1590.0
kw_max_min_ratio         1527.0
max_negative_polarity    1525.0
LDA_04                   1508.0
global_subjectivity      1502.0
kw_avg_x_channel         1502.0
dtype: float64
drift features to drop: [] -> no meaningful drift, nothing dropped


## 2) Ablation — *which change actually helps?*
Each variant is scored by the same fast LightGBM on **identical folds**, measured on **complete train rows only** (test-like). Read the table like this:
- **logloss** (lower is better) is the most stable signal.
- **F1_tuned** = Macro-F1 after class-weight tuning (what we submit).
- **P(better)** = share of 300 bootstrap resamples where the variant beats the baseline. Treat **≥ 0.90** as "probably real", **0.7–0.9** as "maybe", below that as noise. A difference of ~0.003 F1 is within noise on this dataset.
- Keep a change only if logloss improves **and** F1_tuned doesn't drop.

In [4]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(X_fe, y))
cw = (len(y) / (K * np.bincount(y))) ** 0.5
sw_all = cw[y]

def tune_class_weights(P, y, rounds=4, step=0.05):
    w = np.ones(P.shape[1]); cur = f1_score(y, (P * w).argmax(1), average='macro')
    for _ in range(rounds):
        for c in range(P.shape[1]):
            for v in np.arange(0.4, 3.01, step):
                w2 = w.copy(); w2[c] = v
                s = f1_score(y, (P * w2).argmax(1), average='macro')
                if s > cur: cur, w = s, w2
    return w, cur

def probe(Xd, seed=SEED):
    oof = np.zeros((len(Xd), K))
    for k, (tr, va) in enumerate(folds):
        m = lgb.LGBMClassifier(n_estimators=150 if FAST else 2000, learning_rate=0.05, num_leaves=31,
                               min_child_samples=50, subsample=0.8, subsample_freq=1, colsample_bytree=0.6,
                               reg_lambda=3, objective='multiclass', random_state=seed + k, n_jobs=-1, verbose=-1)
        m.fit(Xd.iloc[tr], y[tr], sample_weight=sw_all[tr], eval_set=[(Xd.iloc[va], y[va])],
              eval_sample_weight=[sw_all[va]], callbacks=[lgb.early_stopping(50, verbose=False)])
        oof[va] = m.predict_proba(Xd.iloc[va])
    w, f1t = tune_class_weights(oof[comp], y[comp], rounds=2, step=0.1)
    return dict(oof=oof, w=w, logloss=log_loss(y[comp], oof[comp]), f1_argmax=f1_score(y[comp], oof[comp].argmax(1), average='macro'), f1_tuned=f1t)

def boot_better(ra, rb, n=300, seed=0):
    # P(variant b beats a) over bootstrap resamples of the OOF rows (paired)
    rng = np.random.RandomState(seed); yc = y[comp]; pa = (ra['oof'][comp] * ra['w']).argmax(1); pb = (rb['oof'][comp] * rb['w']).argmax(1)
    d = []
    for _ in range(n):
        i = rng.randint(0, len(yc), len(yc))
        d.append(f1_score(yc[i], pb[i], average='macro') - f1_score(yc[i], pa[i], average='macro'))
    d = np.array(d); return (d > 0).mean(), d.mean()

if RUN_ABLATION:
    variants = {'raw features': X_raw, '+ feature engineering': X_fe}
    if drift_cols: variants['+ FE, drift cols dropped'] = X_fe.drop(columns=drift_cols)
    res = {}; rows = []
    for name, Xd in variants.items():
        t0 = time.time(); res[name] = probe(Xd); r = res[name]
        base = res['raw features']
        pb, dm = (np.nan, np.nan) if name == 'raw features' else boot_better(base, r)
        rows.append(dict(variant=name, logloss=round(r['logloss'], 4), F1_argmax=round(r['f1_argmax'], 4),
                         F1_tuned=round(r['f1_tuned'], 4), dF1_vs_raw=round(dm, 4) if dm == dm else '', **{'P(better)': round(pb, 2) if pb == pb else ''}))
        print(f'{name}: {time.time()-t0:.0f}s')
    abl = pd.DataFrame(rows); display(abl)

raw features: 26s
+ feature engineering: 36s


,variant,logloss,F1_argmax,F1_tuned,dF1_vs_raw,P(better)
0,raw features,1.3324,0.3169,0.3235,,
1,+ feature engineering,1.3306,0.3190,0.3272,0.0037,0.89


**Decision cell** – set these from the table above (defaults follow the safe rule: keep FE and drop drift only if it was detected).

In [5]:
USE_FE = True
if USE_FE:
    X, X_test = X_fe.copy(), Xt_fe.copy()
else:
    X, X_test = X_raw.copy(), Xt_raw.copy()
if DROP_DRIFT and drift_cols:
    X = X.drop(columns=[c for c in drift_cols if c in X.columns]); X_test = X_test[X.columns]
X_test = X_test[X.columns]
print('final feature matrix:', X.shape)

final feature matrix: (37144, 92)


## 3) Full models (GPU boosters + LightGBM), multi-seed

In [6]:
def run_xgb(seed):
    oof = np.zeros((len(X), K)); pred = np.zeros((len(X_test), K))
    for k, (tr, va) in enumerate(folds):
        m = xgb.XGBClassifier(n_estimators=150 if FAST else 3000, learning_rate=0.03, max_depth=6, min_child_weight=5,
            subsample=0.8, colsample_bytree=0.6, reg_lambda=3, reg_alpha=0.5, tree_method='hist',
            device='cuda' if USE_GPU else 'cpu', objective='multi:softprob', eval_metric='mlogloss',
            early_stopping_rounds=100, random_state=seed + k, n_jobs=4)
        m.fit(X.iloc[tr], y[tr], sample_weight=sw_all[tr], eval_set=[(X.iloc[va], y[va])],
              sample_weight_eval_set=[sw_all[va]], verbose=False)
        oof[va] = m.predict_proba(X.iloc[va]); pred += m.predict_proba(X_test) / N_FOLDS
    return oof, pred

def run_cat(seed):
    oof = np.zeros((len(X), K)); pred = np.zeros((len(X_test), K))
    for k, (tr, va) in enumerate(folds):
        m = CatBoostClassifier(iterations=200 if FAST else 4000, learning_rate=0.06, depth=7, l2_leaf_reg=5,
            loss_function='MultiClass', task_type='GPU' if USE_GPU else 'CPU', random_seed=seed + k,
            od_type='Iter', od_wait=150, verbose=False)
        m.fit(X.iloc[tr], y[tr], sample_weight=sw_all[tr], eval_set=(X.iloc[va], y[va]))
        oof[va] = m.predict_proba(X.iloc[va]); pred += m.predict_proba(X_test) / N_FOLDS
    return oof, pred

def run_lgb(seed):
    oof = np.zeros((len(X), K)); pred = np.zeros((len(X_test), K))
    for k, (tr, va) in enumerate(folds):
        m = lgb.LGBMClassifier(n_estimators=150 if FAST else 3000, learning_rate=0.03, num_leaves=48, min_child_samples=30,
            subsample=0.8, subsample_freq=1, colsample_bytree=0.6, reg_lambda=3, reg_alpha=0.5,
            objective='multiclass', random_state=seed + k, n_jobs=-1, verbose=-1)
        m.fit(X.iloc[tr], y[tr], sample_weight=sw_all[tr], eval_set=[(X.iloc[va], y[va])],
              eval_sample_weight=[sw_all[va]], callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[va] = m.predict_proba(X.iloc[va]); pred += m.predict_proba(X_test) / N_FOLDS
    return oof, pred

fns = dict(xgb=run_xgb, cat=run_cat, lgb=run_lgb)
results = {}
for name in MODELS:
    t0 = time.time(); oofs, preds = [], []
    for s in range(N_SEEDS):
        o, p = fns[name](SEED + 100 * s); oofs.append(o); preds.append(p)
        print(f'  {name} seed {s}: F1(argmax, complete rows)={f1_score(y[comp], o[comp].argmax(1), average="macro"):.4f}  '
              f'avg-so-far={f1_score(y[comp], np.mean(oofs,0)[comp].argmax(1), average="macro"):.4f}')
    results[name] = (np.mean(oofs, 0), np.mean(preds, 0))
    print(f'== {name} done in {time.time()-t0:.0f}s')

  xgb seed 0: F1(argmax, complete rows)=0.3155  avg-so-far=0.3155
  xgb seed 1: F1(argmax, complete rows)=0.3191  avg-so-far=0.3167
  xgb seed 2: F1(argmax, complete rows)=0.3170  avg-so-far=0.3159
  xgb seed 3: F1(argmax, complete rows)=0.3166  avg-so-far=0.3154
  xgb seed 4: F1(argmax, complete rows)=0.3154  avg-so-far=0.3147
  xgb seed 5: F1(argmax, complete rows)=0.3164  avg-so-far=0.3157
  xgb seed 6: F1(argmax, complete rows)=0.3167  avg-so-far=0.3159
== xgb done in 232s
  cat seed 0: F1(argmax, complete rows)=0.3154  avg-so-far=0.3154
  cat seed 1: F1(argmax, complete rows)=0.3164  avg-so-far=0.3171
  cat seed 2: F1(argmax, complete rows)=0.3160  avg-so-far=0.3179
  cat seed 3: F1(argmax, complete rows)=0.3145  avg-so-far=0.3181
  cat seed 4: F1(argmax, complete rows)=0.3173  avg-so-far=0.3183
  cat seed 5: F1(argmax, complete rows)=0.3155  avg-so-far=0.3184
  cat seed 6: F1(argmax, complete rows)=0.3153  avg-so-far=0.3200
== cat done in 629s
  lgb seed 0: F1(argmax, complete ro

## 4) Ordinal regressor (A→E are ordered classes)

In [7]:
reg_oof = np.zeros(len(X)); reg_pred = np.zeros(len(X_test))
for k, (tr, va) in enumerate(folds):
    m = lgb.LGBMRegressor(n_estimators=150 if FAST else 3000, learning_rate=0.03, num_leaves=32, min_child_samples=40,
                          subsample=0.8, subsample_freq=1, colsample_bytree=0.6, reg_lambda=3,
                          random_state=SEED + k, n_jobs=-1, verbose=-1)
    m.fit(X.iloc[tr], y[tr], eval_set=[(X.iloc[va], y[va])], callbacks=[lgb.early_stopping(100, verbose=False)])
    reg_oof[va] = m.predict(X.iloc[va]); reg_pred += m.predict(X_test) / N_FOLDS
print('regressor corr with true class index (complete rows):', round(np.corrcoef(reg_oof[comp], y[comp])[0, 1], 4))

regressor corr with true class index (complete rows): 0.414


## 5) Blend, stack, tune class weights, pick the winner
Candidates: **blend** (weighted average of model probabilities) vs **stack** (logistic regression on all model probabilities + the ordinal regressor). The scoreboard shows what each piece contributes.

In [8]:
names = list(results)
# simple blend: grid over weights
best = (-1, None); grid = np.arange(0, 1.01, 0.1)
import itertools
for ws in itertools.product(grid, repeat=len(names)):
    if abs(sum(ws) - 1) > 1e-9: continue
    s = f1_score(y[comp], sum(w * results[n][0] for w, n in zip(ws, names))[comp].argmax(1), average='macro')
    if s > best[0]: best = (s, ws)
blend_oof  = sum(w * results[n][0] for w, n in zip(best[1], names))
blend_test = sum(w * results[n][1] for w, n in zip(best[1], names))

def stack_feats(res_idx, reg):
    return np.hstack([np.log(np.clip(results[n][res_idx], 1e-6, 1)) for n in names] + [reg.reshape(-1, 1)])
S_oof, S_test = stack_feats(0, reg_oof), stack_feats(1, reg_pred)
meta = make_pipeline(StandardScaler(), LogisticRegression(C=0.3, max_iter=3000))
stack_oof = cross_val_predict(meta, S_oof, y, cv=StratifiedKFold(5, shuffle=True, random_state=SEED + 7), method='predict_proba')
stack_test = meta.fit(S_oof, y).predict_proba(S_test)

cands = {f'model: {n}': (results[n][0], results[n][1]) for n in names}
cands['blend'] = (blend_oof, blend_test); cands['stack (+ordinal)'] = (stack_oof, stack_test)
rows = []; tuned = {}
for nm, (o, t) in cands.items():
    w, f1t = tune_class_weights(o[comp], y[comp], rounds=3, step=0.05); tuned[nm] = (w, f1t)
    rows.append(dict(candidate=nm, logloss=round(log_loss(y[comp], o[comp]), 4), F1_argmax=round(f1_score(y[comp], o[comp].argmax(1), average='macro'), 4), F1_tuned=round(f1t, 4)))
board = pd.DataFrame(rows).sort_values('F1_tuned', ascending=False); display(board)

win = max(['blend', 'stack (+ordinal)'], key=lambda n: tuned[n][1])
print('winner for submission:', win)
cwts = tuned[win][0]; oof_final, test_final = cands[win]
print('class weights:', dict(zip(classes, cwts.round(2))))

oc, yc = oof_final[comp], y[comp]
idx = np.random.RandomState(0).permutation(len(yc)); h = len(yc) // 2
w_h, _ = tune_class_weights(oc[idx[:h]], yc[idx[:h]])
print('honest check - weights tuned on half A, scored on half B (complete rows):',
      round(f1_score(yc[idx[h:]], (oc[idx[h:]] * w_h).argmax(1), average='macro'), 4),
      '| untuned:', round(f1_score(yc[idx[h:]], oc[idx[h:]].argmax(1), average='macro'), 4))

,candidate,logloss,F1_argmax,F1_tuned
4,stack (+ordinal),1.2934,0.2722,0.3327
3,blend,1.3184,0.3200,0.3300
1,model: cat,1.3184,0.3200,0.3300
2,model: lgb,1.3240,0.3162,0.3263
0,model: xgb,1.3244,0.3159,0.3255


winner for submission: stack (+ordinal)
class weights: {'A': np.float64(2.75), 'B': np.float64(1.9), 'C': np.float64(1.3), 'D': np.float64(0.9), 'E': np.float64(0.9)}
honest check - weights tuned on half A, scored on half B (complete rows): 0.3209 | untuned: 0.2702


## 6) Write submission

In [9]:
final = (test_final * cwts).argmax(1)
sub = pd.DataFrame({'id': test['id'], 'target_popularity': [classes[i] for i in final]})
sub.to_csv('submission.csv', index=False)
print(sub.shape, 'unique ids ok:', sub.id.nunique() == len(test))
print(pd.concat([sub.target_popularity.value_counts(normalize=True).sort_index().rename('pred'),
                 pd.Series(y).map(dict(enumerate(classes))).value_counts(normalize=True).sort_index().rename('train')], axis=1).round(3))
sub.head()

(2500, 2) unique ids ok: True
    pred  train
A  0.088  0.046
B  0.109  0.106
C  0.169  0.195
D  0.382  0.364
E  0.252  0.289


,id,target_popularity
0,9516,C
1,35906,D
2,20105,D
3,35214,E
4,3184,A
